In [1]:
from pathlib import Path
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np

In [2]:
class IntelDataset(Dataset):

    def __init__(self, root_dir, transform=None):

        self.root_dir = Path(root_dir)
        self.transform = transform

        self.classes = sorted(
            [folder.name for folder in self.root_dir.iterdir()
             if folder.is_dir()]
        )

        self.class_to_idx = {
            cls_name: idx
            for idx, cls_name in enumerate(self.classes)
        }

        self.image_paths = []
        self.labels = []

        for class_name in self.classes:

            class_dir = self.root_dir / class_name

            for image_path in class_dir.glob("*"):

                self.image_paths.append(image_path)
                self.labels.append(
                    self.class_to_idx[class_name]
                )

    def __len__(self):

        return len(self.image_paths)

    def __getitem__(self, idx):

        image_path = self.image_paths[idx]

        image = Image.open(image_path).convert("RGB")

        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [3]:
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
print("Dataset Size:", len(dataset))
print()

print("Classes:")
print(dataset.classes)

print()

print("Class Mapping:")
print(dataset.class_to_idx)

In [ ]:
images, labels = next(iter(train_loader))

print("Batch Image Shape:")
print(images.shape)

print()

print("Batch Labels:")
print(labels)

In [ ]:
fig, axes = plt.subplots(
    2,
    3,
    figsize=(12,8)
)

shown_classes = set()

for idx in range(len(dataset)):

    image, label = dataset[idx]

    class_name = dataset.classes[label]

    if class_name not in shown_classes:

        row = len(shown_classes) // 3
        col = len(shown_classes) % 3

        image = image.permute(1,2,0).numpy()

        image = (
            image * np.array([0.229,0.224,0.225])
            + np.array([0.485,0.456,0.406])
        )

        image = np.clip(image,0,1)

        axes[row,col].imshow(image)
        axes[row,col].set_title(class_name)
        axes[row,col].axis("off")

        shown_classes.add(class_name)

    if len(shown_classes) == len(dataset.classes):
        break

plt.tight_layout()
plt.show()

In [ ]:
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(
    4,
    4,
    figsize=(10,10)
)

for i, ax in enumerate(axes.flatten()):

    image = images[i].permute(1,2,0).numpy()

    image = (
        image * np.array([0.229,0.224,0.225])
        + np.array([0.485,0.456,0.406])
    )

    image = np.clip(image,0,1)

    ax.imshow(image)

    ax.set_title(
        dataset.classes[labels[i]]
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)